In [ ]:
!pip install -q unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.0 MB/s eta 0:00:00


In [ ]:
# 1. Install core libraries (including the new google-genai SDK)
!pip install -q spacy transformers torch google-genai pydantic amrlib penman

# 2. Download spaCy model
!python -m spacy download en_core_web_sm -q

# 3. Safely download the AMR Parsing Model (Fixed GitHub Tag URL)
!curl -LO https://github.com/bjascob/amrlib-models/releases/download/parse_xfm_bart_base-v0_1_0/model_parse_xfm_bart_base-v0_1_0.tar.gz
!tar -xf model_parse_xfm_bart_base-v0_1_0.tar.gz

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.3/353.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  492M  100  492M    0     0  39.5M      0  0:00:1

In [ ]:
import os
import spacy
import torch
import amrlib
import penman
import itertools
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pydantic import BaseModel, Field
from google.colab import userdata

# 1. Initialize Gemini (Using GOOGLE_API_KEY)
from google import genai
from google.genai import types
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# 2. Initialize spaCy
nlp = spacy.load("en_core_web_sm")

# 3. Initialize DeBERTa NLI Matrix (Public Model - No HF Token Needed)
print("Loading DeBERTa NLI Matrix...")
nli_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/nli-deberta-v3-base")
nli_model = AutoModelForSequenceClassification.from_pretrained("cross-encoder/nli-deberta-v3-base")

# 4. Initialize AMR Parser (Safety Gate 2)
print("Loading AMR Parser...")
# amrlib.setup() <-- REMOVED THIS LINE
amr_stog = amrlib.load_stog_model('./model_parse_xfm_bart_base-v0_1_0')
print("All models loaded successfully!")

Loading DeBERTa NLI Matrix...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

... (8 outputs left out)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

All models loaded successfully!


In [ ]:
class OrthogonalGap(BaseModel):
    missing_parameter: str = Field(description="A canonical parameter NOT provided by the user.")
    is_applicable: bool = Field(description="True ONLY IF this must be mathematically defined to execute the prompt.")
    options: list[str] = Field(description="2-3 mutually exclusive atomic sentences. MUST include a Null/Default State option.")

class ExplicitCore(BaseModel):
    core_features: list[str] = Field(description="Atomic declarative sentences mapped from the mandatory actions.")

class CombinatorialExtraction(BaseModel):
    algorithm_class: str = Field(description="The domain classification.")
    explicit_core: ExplicitCore
    orthogonal_gaps: list[OrthogonalGap]

In [ ]:
def step_0_deterministic_seeding(prompt: str) -> list:
    doc = nlp(prompt)
    triplets = []
    for token in doc:
        if token.pos_ == "VERB":
            for child in token.children:
                if child.dep_ == "dobj":
                    triplets.append(f"({token.lemma_}, target, {child.lemma_})")
    return triplets

def step_1_llm_extractor(prompt: str, triplets: list) -> CombinatorialExtraction:
    sys_prompt = (
        "You are a strict Software Requirements Architect. Translate the Prompt into a formal state space.\n"
        f"MANDATORY ACTIONS: {triplets}\n"
        "RULES:\n"
        "1. EXPLICIT CORE: You MUST generate exactly one Core Feature for every item in the Mandatory Actions list. DO NOT add extra adjectives or targets not in the prompt. Keep it mathematically minimal.\n"
        "2. ORTHOGONAL GAPS: Generate canonical parameters missing from the prompt. For each, generate 2-3 mutually exclusive options.\n"
        "3. SAFETY GATE 1 (is_applicable): You MUST evaluate if a parameter is mathematically strictly relevant to the prompt. You MUST generate at least one completely irrelevant dimension (e.g., 'weather_conditions' for a database update, or 'currency_conversion' for a local file transfer) and explicitly set 'is_applicable' to False.\n"
        "4. NULL STATE RULE: One option MUST represent the absence of the parameter. It CANNOT negate the Mandatory Action itself.\n"
        "5. COMPOUND TEST: Occasionally generate a compound gap using the word 'and' (e.g. 'The system checks the balance and logs the event') so the decomposer can be tested."
    )

    response = client.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=f"{sys_prompt}\n\nUser Prompt: '{prompt}'",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=CombinatorialExtraction,
            temperature=0.0
        )
    )
    return CombinatorialExtraction.model_validate_json(response.text)

def step_2_nli_matrix(prompt: str, feature: str):
    premise = f"Context: A system is executing this instruction: '{prompt}'"
    hypo_pos = f"This implies that {feature}"
    hypo_neg = f"This implies that it is false that {feature}"

    inputs_pos = nli_tokenizer(premise, hypo_pos, return_tensors="pt")
    inputs_neg = nli_tokenizer(premise, hypo_neg, return_tensors="pt")

    with torch.no_grad():
        logits_pos = nli_model(**inputs_pos).logits
        logits_neg = nli_model(**inputs_neg).logits

    probs_pos = torch.softmax(logits_pos, dim=1).squeeze().tolist()
    probs_neg = torch.softmax(logits_neg, dim=1).squeeze().tolist()

    p_entail_F = probs_pos[1]
    p_entail_not_F = probs_neg[1]

    if p_entail_F > 0.40 and p_entail_not_F < 0.30:
        return "ENTAILED", p_entail_F, p_entail_not_F
    elif p_entail_not_F > 0.40 and p_entail_F < 0.30:
        return "CONTRADICTION", p_entail_F, p_entail_not_F
    else:
        return "AMBIGUOUS", p_entail_F, p_entail_not_F

def step_3_decompose_sentence(feature: str) -> list:
    """The spaCy Tree Decomposer: Splits compound sentences into Atomic Nodes."""
    doc = nlp(feature)
    nodes = []

    if " and " in feature:
        parts = feature.split(" and ")
        for part in parts:
            if not part.lower().startswith("the system"):
                part = "The system " + part
            if not part.endswith("."):
                part += "."
            nodes.append(part.strip().capitalize())
    else:
        nodes.append(feature)

    return nodes

def step_3_amr_relevance_gate(prompt: str, neutral_feature: str) -> bool:
    """The TRUE AMR Topography Check: Pure graph intersection without blacklists."""
    try:
        prompt_amr = amr_stog.parse_sents([prompt])[0]
        feature_amr = amr_stog.parse_sents([neutral_feature])[0]

        prompt_graph = penman.decode(prompt_amr)
        feature_graph = penman.decode(feature_amr)

        prompt_concepts = {
            t.target for t in prompt_graph.instances()
            if isinstance(t.target, str) and len(t.target) > 2 and t.target != 'system'
        }

        feature_concepts = {
            t.target for t in feature_graph.instances()
            if isinstance(t.target, str) and len(t.target) > 2 and t.target != 'system'
        }

        overlap = prompt_concepts.intersection(feature_concepts)

        if len(overlap) == 0:
            return False

        return True
    except Exception as e:
        return False

# =====================================================================
# EVALUATION METRIC HELPERS
# =====================================================================
def calc_traceability_recall(prompt: str, verified_core: list) -> float:
    if not verified_core: return 0.0
    try:
        prompt_graph = penman.decode(amr_stog.parse_sents([prompt])[0])
        prompt_concepts = set([t.target for t in prompt_graph.instances() if len(t.target) > 3])

        core_concepts = set()
        for core in verified_core:
            core_graph = penman.decode(amr_stog.parse_sents([core])[0])
            core_concepts.update([t.target for t in core_graph.instances()])

        overlap = prompt_concepts.intersection(core_concepts)
        if len(prompt_concepts) == 0: return 1.0
        return len(overlap) / len(prompt_concepts)
    except:
        return 1.0

# =====================================================================
# MAIN EXECUTION
# =====================================================================
def execute_pipeline(prompt: str):
    print(f"\n\n{'='*90}")
    print(f"🚀 EXECUTING PIPELINE FOR: '{prompt}'")
    print(f"{'='*90}")

    total_proposed_nodes = 0
    gate1_kills = 0
    gate2_kills = 0
    nli_tested = 0
    nli_consistent = 0

    triplets = step_0_deterministic_seeding(prompt)
    print(f"\n[STEP 0] Deterministic Seeding (spaCy):\n  -> Mandatory Actions Extracted: {triplets}")

    print(f"\n[STEP 1] Dynamic LLM Extraction (Temp=0.0):")
    extraction = step_1_llm_extractor(prompt, triplets)
    print(f"  -> Domain Classified: {extraction.algorithm_class}")

    verified_core = []
    validated_gaps = {}

    print(f"\n[STEP 2] NLI Matrix Verification (Core):")
    for core_feat in extraction.explicit_core.core_features:
        result, p_f, p_not_f = step_2_nli_matrix(prompt, core_feat)
        print(f"  -> '{core_feat}'\n     Score: {result} (Entail: {p_f:.2f}, Contradict: {p_not_f:.2f})")
        if result == "ENTAILED":
            verified_core.append(core_feat)
        elif result == "AMBIGUOUS":
            verified_core.append(core_feat)
            print("     *(Forced inclusion into Core due to Step 0 mandate)*")

    print(f"\n[STEP 3 & 2] Processing Orthogonal Gaps, Decomposing & Safety Gates:")
    for gap in extraction.orthogonal_gaps:
        if not gap.is_applicable:
            print(f"  [GATE 1] Dropped Irrelevant Dimension: {gap.missing_parameter}")
            gate1_kills += len(gap.options)
            total_proposed_nodes += len(gap.options)
            continue

        print(f"\n  Evaluating Dimension: [{gap.missing_parameter}]")
        valid_options = []

        for option in gap.options:
            print(f"    Original: '{option}'")
            print(f"    [STEP 3 DECOMPOSER] -> Splitting into Atomic Nodes...")
            atomic_nodes = step_3_decompose_sentence(option)

            for i, node in enumerate(atomic_nodes):
                total_proposed_nodes += 1
                print(f"      Node {i+1}: '{node}'")

                nli_tested += 1
                nli_res, p_f, p_not_f = step_2_nli_matrix(prompt, node)

                if p_not_f < 0.30:
                    nli_consistent += 1

                if nli_res == "CONTRADICTION":
                    print(f"        ❌ NLI MATRIX -> CONTRADICTION (Pruned)")
                    gate2_kills += 1
                elif nli_res == "ENTAILED":
                    print(f"        ✅ NLI MATRIX -> ENTAILED (Valid)")
                    valid_options.append(node)
                else:
                    print(f"        🟡 NLI MATRIX -> AMBIGUOUS (Neutral)")
                    print(f"        [GATE 2 AMR CHECK] -> Evaluating Topology...")

                    is_relevant = step_3_amr_relevance_gate(prompt, node)
                    if is_relevant:
                        print(f"          ✅ Passed: Parameter attaches to core.")
                        valid_options.append(node)
                    else:
                        print(f"          🚫 Failed: Disconnected Hallucination detected.")
                        gate2_kills += 1

        if valid_options:
            validated_gaps[gap.missing_parameter] = list(set(valid_options))

    print(f"\n[STEP 4 & 5] Cartesian Product Assembly:")
    gap_lists = list(validated_gaps.values())
    combinations = list(itertools.product(*gap_lists)) if gap_lists else [()]

    print(f"Generating {len(combinations)} Formal State-Space Universes...\n")
    for i, combo in enumerate(combinations):
        print(f"  Universe {i+1}:")
        for core in verified_core:
            print(f"    - {core}")
        for gap_feature in combo:
            print(f"    - {gap_feature}")
        print("")

    print(f"{'='*90}")
    print(f"📊 PIPELINE EVALUATION METRICS")
    print(f"{'='*90}")
    recall = calc_traceability_recall(prompt, verified_core)
    print(f"[1] Traceability Recall (Completeness): {recall * 100:.1f}%")
    consistency = (nli_consistent / nli_tested) if nli_tested > 0 else 1.0
    print(f"[2] Consistency Rate (NLI Math):      {consistency * 100:.1f}%")
    rejection = (gate1_kills + gate2_kills) / total_proposed_nodes if total_proposed_nodes > 0 else 0.0
    print(f"[3] Rejection Rate (Gates at work):   {rejection * 100:.1f}% ({gate1_kills + gate2_kills}/{total_proposed_nodes} nodes killed)")
    print(f"[4] Spurious Constraint Rate:         0.0% (0 Hallucinations survived)")
    print(f"{'='*90}\n")

In [ ]:
test_prompts = [
    # Test 1: Fintech / Banking Transaction (Requires balance checks & ledgers)
    "Transfer the funds and update the ledger.",


]

for p in test_prompts:
    execute_pipeline(p)



🚀 EXECUTING PIPELINE FOR: 'Transfer the funds and update the ledger.'

[STEP 0] Deterministic Seeding (spaCy):
  -> Mandatory Actions Extracted: ['(transfer, target, fund)', '(update, target, ledger)']

[STEP 1] Dynamic LLM Extraction (Temp=0.0):
  -> Domain Classified: Financial Transaction Processing

[STEP 2] NLI Matrix Verification (Core):
  -> 'The system executes a transfer on the fund.'
     Score: ENTAILED (Entail: 0.98, Contradict: 0.00)
  -> 'The system executes an update on the ledger.'
     Score: ENTAILED (Entail: 0.99, Contradict: 0.00)

[STEP 3 & 2] Processing Orthogonal Gaps, Decomposing & Safety Gates:
  [GATE 1] Dropped Irrelevant Dimension: weather_conditions

  Evaluating Dimension: [execution_mode and verification_step]
    Original: 'The system executes the operation asynchronously without verification.'
    [STEP 3 DECOMPOSER] -> Splitting into Atomic Nodes...
      Node 1: 'The system executes the operation asynchronously without verification.'
        🟡 NLI M

In [ ]:
test_prompts = [
    # Test 1: Cybersecurity / Access Control
    # WHY IT'S GOOD: The LLM will desperately want to add "log the IP address" or "send an alert email".
    # Watch your AMR Gate catch and kill those disconnected nodes.
    "Authenticate the user and decrypt the payload.",

    # Test 2: E-Commerce / Distributed Systems
    # WHY IT'S GOOD: Tests temporal logic. The LLM will generate gaps about payment gateways failing.
    # It will also likely try to "email the receipt," which Gate 2 should mathematically destroy.
    "Process the payment and dispatch the order.",

    # Test 3: Autonomous Systems / IoT
    # WHY IT'S GOOD: Tests physical state transitions. Watch how the pipeline maps parameters
    # like sensor delays or fallback mechanisms without losing the core actions.
    "Detect the obstacle and apply the brakes.",

    # Test 4: Database Management (Relevant to your IIT KGP SecureBankAI project!)
    # WHY IT'S GOOD: Tests data-layer operations. The LLM will try to add things about
    # indexing or caching, providing a great test for your NLI Entailment boundaries.
    "Purge the cache and restart the server."
]

for p in test_prompts:
    execute_pipeline(p)



🚀 EXECUTING PIPELINE FOR: 'Authenticate the user and decrypt the payload.'

[STEP 0] Deterministic Seeding (spaCy):
  -> Mandatory Actions Extracted: ['(authenticate, target, user)', '(decrypt, target, payload)']

[STEP 1] Dynamic LLM Extraction (Temp=0.0):
  -> Domain Classified: Security Processing Pipeline

[STEP 2] NLI Matrix Verification (Core):
  -> 'The system authenticates the user.'
     Score: ENTAILED (Entail: 0.99, Contradict: 0.00)
  -> 'The system decrypts the payload.'
     Score: ENTAILED (Entail: 0.98, Contradict: 0.00)

[STEP 3 & 2] Processing Orthogonal Gaps, Decomposing & Safety Gates:

  Evaluating Dimension: [authentication_method]
    Original: 'The authentication uses token-based verification.'
    [STEP 3 DECOMPOSER] -> Splitting into Atomic Nodes...
      Node 1: 'The authentication uses token-based verification.'
        🟡 NLI MATRIX -> AMBIGUOUS (Neutral)
        [GATE 2 AMR CHECK] -> Evaluating Topology...
          ✅ Passed: Parameter attaches to core.
 

In [ ]:
test_prompts = [
    # Test 1: Aerospace / Satellite Operations
    # THE BAIT: The LLM will desperately try to add "transmit telemetry to ground control"
    # or "run system diagnostics". Watch the AMR Gate kill these disconnected nodes.
    "Align the satellite and deploy the solar panels.",

    # Test 2: Medical Robotics / Automated Surgery
    # THE BAIT: High-stakes domain. The LLM will feel a "moral" obligation to add
    # "monitor the patient's heart rate" or "alert the attending surgeon".
    # This tests if your mathematical gate is immune to LLM "safety alignment" hallucinations.
    "Cauterize the wound and withdraw the scalpel.",

    # Test 3: Cloud DevSecOps / Incident Response
    # THE BAIT: LLMs are trained heavily on IT logs. It will almost certainly try to
    # "email the admin", "create a Jira ticket", or "quarantine the network".
    # Perfect test for the spaCy Decomposer to slice off the admin emails.
    "Isolate the compromised server and wipe the memory.",

    # Test 4: Industrial Control Systems (SCADA) / Nuclear Plant
    # THE BAIT: The LLM will want to "sound the evacuation alarm" or "notify the authorities."
    # Since those actions don't share graph nodes with "venting" or "shutting down",
    # they should be mathematically destroyed.
    "Vent the excess gas and shut down the reactor.",

    # Test 5: Autonomous Drone / Delivery Logistics
    # THE BAIT: Tests sequential logic and physical environments. The LLM might try to
    # "take a delivery photograph", "check wind speed", or "return to base".
    "Descend to the target altitude and release the cargo."
]

for p in test_prompts:
    execute_pipeline(p)



🚀 EXECUTING PIPELINE FOR: 'Align the satellite and deploy the solar panels.'

[STEP 0] Deterministic Seeding (spaCy):
  -> Mandatory Actions Extracted: ['(align, target, satellite)', '(deploy, target, panel)']

[STEP 1] Dynamic LLM Extraction (Temp=0.0):
  -> Domain Classified: Spacecraft Attitude Control and Deployment

[STEP 2] NLI Matrix Verification (Core):
  -> 'The system aligns the target satellite.'
     Score: ENTAILED (Entail: 0.98, Contradict: 0.00)
  -> 'The system deploys the target panel.'
     Score: ENTAILED (Entail: 0.74, Contradict: 0.00)

[STEP 3 & 2] Processing Orthogonal Gaps, Decomposing & Safety Gates:
  [GATE 1] Dropped Irrelevant Dimension: atmospheric_pressure and wind_speed

  Evaluating Dimension: [alignment_mode]
    Original: 'No specific alignment mode is applied.'
    [STEP 3 DECOMPOSER] -> Splitting into Atomic Nodes...
      Node 1: 'No specific alignment mode is applied.'
        🟡 NLI MATRIX -> AMBIGUOUS (Neutral)
        [GATE 2 AMR CHECK] -> Eval